In [ ]:
import json
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from glob import glob
import os
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import Javascript, display
from IPython.display import HTML, Image
from pathlib import Path

In [ ]:
def read_results(path):
    #print(f"\tTrying to read {path}")
    if os.path.exists(path):
        with open(path) as f:
            results = json.load(f)
        return results
    else:
        print(f"Cannot find file {path}")
        return None

lang_map = {"eng": "en",
            "fra": "fr",
            "fin": "fi",
            "zho": "zh",
            "ita": "it",
            "deu": "de",
            "vie": "vi",
            "jpn": "ja",
            "ukr": "uk",
            "ell": "el",
            "hin": "hi",
            "bul": "bg",
            "fas": "fa"}

In [ ]:
def parse_mteb_results(models, dataset):
    if dataset == "RTE3-multi":
        return parse_rte(models, dataset)
    print("Reading MTEB results...")
    # path where the results are
    mteb_results_path=lambda x: [i for i in glob(f"../mteb_out/{x}/results/{x}/*/{dataset}.json")]
    results = {}
    for model in models:
        #print(f"In {model}")
        if model == "gemini-embedding-001":
            model_to_read = "google__gemini-embedding-001"
        else:
            model_to_read = model
        mteb_results_path_ = mteb_results_path(model_to_read)
        if mteb_results_path_ == []:
            print(f"\tModel {model} missing results from mteb")
            continue
        if len(mteb_results_path_) != 1:
            # the dataset revision has changed, using the first results
            #print(f"\t{len(mteb_results_path_)} evaluation files found for {model}. Selecting first results.")
            mteb_results_path_ = [mteb_results_path_[0]]
        
        d = read_results(mteb_results_path_[0])  # [0] just to remove nesting
        for subsplit in d["scores"]["test"]:
            #print("in subsplit", subsplit)
            if not f'{subsplit["hf_subset"]}' in results.keys():
                results[f'{subsplit["hf_subset"]}'] = {}
            results[f'{subsplit["hf_subset"]}'][f'{model}']= subsplit["main_score"]
    return results

def parse_rte(models, dataset):
    results = {}
    langs_rte = ["en", "fr", "it", "de"]
    mteb_results_path="../mteb_out/rte_crawled.json"
    j = read_results(mteb_results_path)
    for m in models:
        m_ = m.replace("__", "/")
        if "gemini" in m_:
            m_ = "google/gemini-embedding-001"
        for l in langs_rte:
            if l not in results.keys():
                results[l] = {}
            #print(f"{m_}_{l}")
            results[l][m] = j[f"{m_}_{l}"]
    return results

def parse_NN_results(models, dataset, splits, prompt=False, ks = ["5","10","20", "50"], percentages=["1%", "2%", "5%", "10%", "20%"]):
    #print("Reading NN-results...")
    NN_results_path= lambda model, data: f"../knn_results{'_with_prompt' if prompt else ''}/{data}/{model}.json"
    knns_avg = {}
    knns_avg_perc = {}
    knns = {}
    knns_perc = {}
    r2s = {}
    for m in models:
        #print(f"now in {m}")
        if "openai" in m:
            m_ = m.replace("openai__","")
        else:
            m_ = m
        for s in splits:
            #print(f"in split {s}")
            if s == "default":
                path = NN_results_path(m_, dataset)
            else:
                if dataset == "tatoeba":
                    # remap splits to match hf format
                    s_ = lang_map[s.split("-")[1]]+"-"+lang_map[s.split("-")[0]]
                    path = NN_results_path(m_, dataset+":"+s_)
                else:
                    path = NN_results_path(m_, dataset+":"+s)
            if not os.path.isfile(path):
                print(f"\tmissing results for {path}")
                continue
            else:
                j = read_results(path)
                #print(j)
            if s not in knns_avg.keys():
                knns_avg[s] = {}
                knns_avg_perc[s] = {}
                r2s[s] = {}
                #knns[s] = {}
                #knns_perc[s] = {}
            knns_avg[s][m] = j["knn_avg"]
            knns_avg_perc[s][m] = j["knn_avg_percentage"]
            r2s[s][m] = j["r2_affine"]
            for k, p in zip(ks, percentages):
                if k not in knns.keys():
                    knns[k] = {}
                    knns_perc[p] ={}
                if s not in knns[k].keys():
                    knns[k][s] = {}
                    knns_perc[p][s] = {}
                knns[k][s][m] = j["knn_scores"][k] #{k:v for k,v in j["knn_scores"]}
                knns_perc[p][s][m] = j["knn_scores_percentage"][p] #{k:v for k,v in j["knn_scores_percentage"]}
           
    return knns_avg, knns_avg_perc, knns, knns_perc, r2s



def color_text(s):
    return f"\x1b[31m{s}\x1b[0m"

def analysis(dataset, df, score, against,split="default", pprint=False):
    collect_results = []
    if isinstance(score, str):
        score = [score]
    for s in score:
        x = df[s].tolist()
        y = df[against].tolist()
        pr = pearsonr(x,y)
        sp = spearmanr(x,y)
        p_value = round(pr.pvalue,5)
        p_value = color_text(p_value) if p_value <= 0.05 else p_value
        sp_value = round(sp.pvalue, 5)
        sp_value = color_text(sp_value) if sp_value <= 0.05 else sp_value
        if pprint:
            print(f"\t Statistic: {s}\t(support {len(x)})\t Result: {round(sp.statistic, 3)} (p-value {sp_value})")
            #print(f"\t Statistic: {s} (support {len(x)})\t\tP-Result: {round(pr.statistic, 3)} (p-value {p_value})  \n\
            #    \t\t\t\tS-Result: {round(sp.statistic, 3)} (p-value {sp_value})")

        collect_results.append({"dataset": dataset, "split":split, "statistic":s, "support": len(x), "pr": (pr.statistic, pr.pvalue), "sp":(sp.statistic, sp.pvalue)})
    return collect_results

def make_table(dataset, mteb_scores, dict_of_results, pprint=False):
    results_knn = []
    results_bss = []
    for split in mteb_scores.keys():
        print(split)
        dfs = []
        df_mteb = pd.DataFrame([mteb_scores[split]]).T.rename(columns={0:"mteb_score"})
        for name, result in dict_of_results.items():
            df = pd.DataFrame([result[split]]).T.rename(columns={0:name})
            dfs.append(df)
        df = pd.concat([df_mteb] + dfs,  axis=1)
        df = df.reset_index().rename(columns={"index":"model"})
        df = df.dropna()   # drops all NaN values --> i.e. do not analyse unless all results exist
        r1 = analysis(dataset, df, [i for i in dict_of_results.keys() if "knn" in i or  "r2" in i ], "mteb_score", split=split, pprint=pprint)
        r2 = analysis(dataset, df, [i for i in dict_of_results.keys() if "gini" in i or "peak" in i], "mteb_score", split=split, pprint=pprint)
        results_knn.append(r1)
        results_bss.append(r2)
    return results_knn, results_bss

In [ ]:
models = [os.path.basename(m) for m in glob("path_to_models")] + ["openai__text-embedding-3-small","openai__text-embedding-3-large"]

dataset_map ={"ARCChallenge" : "arcchallenge",
              "RTE3-multi": "rte3-multi__contradiction",
              "SummEvalSummarization.v2": "summeval",
              "Tatoeba": "tatoeba",
              "WebFAQRetrieval": "web-faq-bitext"}

In [ ]:
results = {}
for d, d_ in dataset_map.items():
    results[d] = []
    mteb_scores = parse_mteb_results(models, d)
    knns_avg, knns_avg_perc, knns, knns_perc, r2s = parse_NN_results(models, d_, [k for k in mteb_scores.keys()], prompt=False)
    #print(mteb_scores)
    #print(knns_avg)
    #print(knns_perc)
    #print(d)
    #print(knns)
    df, _ = make_table(d, mteb_scores, {"knn_avg": knns_avg, 
                                        "knn_avg_%": knns_avg_perc, 
                                        "knn_1%": knns_perc["1%"], 
                                        "knn_2%": knns_perc["2%"],
                                        "knn_5%": knns_perc["5%"],
                                        "knn_10%": knns_perc["10%"],
                                        "knn_5": knns["5"],
                                        "knn_10": knns["10"],
                                        "knn_20": knns["20"],
                                        "knn_50": knns["50"]}
                                        , pprint=False)

    
    for row in df:
        stats=[]
        rs= []
        for a in row:
            r = a["sp"][0]
            s = a["statistic"]
            rs.append(r)
            stats.append(s)
        #print(rs)
        #print(stats)
    
        best_rs = np.argsort(rs)[::-1]  # see analysis/README.md
        results[d].append(np.array(stats)[best_rs])
            
    #display(df)
print(results)
        
        

In [ ]:
from collections import defaultdict
# evaluate which is best by simply giving scores (borda) to each metric


def borda_scores(rankings):
    scores = defaultdict(int)
    for r in rankings:
        m = len(r)
        for pos, item in enumerate(r):
            scores[item] += (m - 1 - pos)
    return dict(scores)

def average_rank(rankings):
    total_pos = defaultdict(int)
    counts = defaultdict(int)
    for r in rankings:
        for pos, item in enumerate(r, start=1):
            total_pos[item] += pos
            counts[item] += 1
    return {item: total_pos[item]/counts[item] for item in total_pos}



for d in results.keys():
    rankings = results[d]
    scores = borda_scores(rankings)
    avgr = average_rank(rankings)
    print(f"{d} Borda scores (higher is better):", dict(sorted(scores.items(), key=lambda x: -x[1])))
    print(f"{d} Average rank (lower is better):", dict(sorted(avgr.items(), key=lambda x: x[1])))
    print("")

In [ ]:
dataset_size = {"arcchallenge":937,
                "tatoeba": 5000,
                "web-faq-multi": 5000,
                "summeval": 80,
                "rte-multi":90
                }
for d,s in dataset_size.items():
    print(f"{d}: {s*0.1}")